In [9]:
import numpy as np
from fury import window, actor

# Assuming your files are named exactly as follows:
from data import data, affine, labels, bvals, bvecs, vox_size
from env_main import UnifiedTractographyEnv
from agent import EuDXAgent,EnhancedEuDXAgent
from dipy.tracking import utils
from dipy.viz import colormap


In [2]:
print("Initializing Unified Tractography Environment...")
env = UnifiedTractographyEnv(
    data=data,
    affine=affine,
    labels=labels,
    bvals=bvals,
    bvecs=bvecs,
    vox_size=vox_size,
    step_size=0.5,
    max_steps=500,
    fa_threshold=0.25,
    max_curvature_deg=45
)

Initializing Unified Tractography Environment...


<B>Vsualizing enviroment</B>

In [3]:
scene=window.Scene()
seed_mask=labels==2
env.render_wm_surface(scene=scene)
# env.render_seeds_mask(scene=scene)
env.render_bval_bvec(scene,seed_mask)
window.show(scene,size=(800,800))

In [14]:
agent = EuDXAgent(env)


In [10]:
EnhancedEuDXAgent = EnhancedEuDXAgent(env)

In [18]:
seed_indices=np.argwhere(labels == 2)

In [14]:
obs, _ =env.reset()
print(obs)

[39.        41.        39.         0.         0.         0.
  0.7858934  0.       ]


<B>Select Seeds</B>

In [7]:
seed_indices = np.argwhere(labels == 2)
print(f"Found {len(seed_indices)} potential seed points.")

Found 505 potential seed points.


In [5]:
seeds=utils.seeds_from_mask(mask=(labels==2),affine=affine,density=(2,2,2))

In [15]:
all_streamlines=[]
all_streamlines=agent.track_all(seeds)

In [16]:
scene.clear()
env.render_streamlines(scene=scene,streamlines=all_streamlines)
window.show(scene,size=(800,800))

In [17]:
from dipy.io.streamline import save_trk
from data import hardi_img
from dipy.io.stateful_tractogram import Space, StatefulTractogram
sft = StatefulTractogram(all_streamlines, hardi_img, Space.RASMM)
save_trk(sft, "trk/tractogram_EuDXagent.trk", all_streamlines)

In [35]:
print(f"Found {len(seeds)} potential seed points.")

Found 4040 potential seed points.


In [ ]:
print(seed_indices[:5],seeds[:5])


[[38 38 34]
 [38 41 33]
 [38 42 33]
 [38 43 33]
 [39 38 34]] [[ -4.5 -44.5   7.5]
 [ -3.5 -44.5   7.5]
 [ -4.5 -43.5   7.5]
 [ -3.5 -43.5   7.5]
 [ -4.5 -44.5   8.5]]


In [45]:
inv_affine = np.linalg.inv(affine)
for seed in seeds:
        # Convert seed to voxel space
        current_pos_world = np.array(seed)
        current_pos_vox = (np.append(seed, 1) @ inv_affine.T)[:3]
print(seeds[:5])
seed_indices=seeds

[[ -4.5 -44.5   7.5]
 [ -3.5 -44.5   7.5]
 [ -4.5 -43.5   7.5]
 [ -3.5 -43.5   7.5]
 [ -4.5 -44.5   8.5]]


In [46]:
num_seeds_to_track =  len(seed_indices)
all_streamlines = []

print(f"Tracking {num_seeds_to_track} streamlines...")
for i in range(num_seeds_to_track):
    seed = seed_indices[i]
    streamline = agent.generate_streamline(seed)
    
    if len(streamline) > 1:
        all_streamlines.append(streamline)
print(f"Genrated {len(all_streamlines)} streamlines...")

Tracking 4040 streamlines...
Genrated 0 streamlines...


In [23]:

def custom_tracker(seeds, csa_peaks, affine, step_size=0.5, gfa_thresh=0.25, max_steps=500):
    """
    Manual implementation of a deterministic peak-following tracker.
    """
    all_streamlines = []
    gfa = csa_peaks.gfa
    peaks = csa_peaks.peak_dirs  # Shape (X, Y, Z, 5, 3)
    
    # Invert affine to move from world coordinates (seeds) to voxel indices
    inv_affine = np.linalg.inv(affine)

    for seed in seeds:
        # Convert seed to voxel space
        current_pos_world = np.array(seed)
        current_pos_vox = (np.append(seed, 1) @ inv_affine.T)[:3]
        
        streamline = [current_pos_world.copy()]
        prev_dir = None
        
        for _ in range(max_steps):
            # 1. Get voxel index
            idx = tuple(np.floor(current_pos_vox).astype(int))
            
            # 2. Stopping Criteria
            # Check boundaries
            if any(i < 0 or i >= s for i, s in zip(idx, gfa.shape)):
                break
            # Check GFA
            if gfa[idx] < gfa_thresh:
                break
                
            # 3. Get local peaks
            local_peaks = peaks[idx]  # Usually 5 peaks per voxel
            
            # 4. Determine direction
            if prev_dir is None:
                # First step: use the primary peak (peak 0)
                best_dir = local_peaks[0]
            else:
                # Follow-up steps: pick peak most aligned with prev_dir
                # Use absolute dot product because fiber orientations are axial
                dots = np.dot(local_peaks, prev_dir)
                best_peak_idx = np.argmax(np.abs(dots))
                best_dir = local_peaks[best_peak_idx]
                
                # Ensure we don't go backwards (keep angle < 90 deg)
                if np.dot(best_dir, prev_dir) < 0:
                    best_dir = -best_dir
                
                # Curvature check: angle > 45 degrees
                cos_sim = np.dot(best_dir, prev_dir)
                if cos_sim < np.cos(np.deg2rad(45)):
                    break

            # 5. Move in World Space (EuDX logic)
            # EuDX moves in the direction of the peak, scaled by step_size
            # We must convert the direction to world space for the next world point
            # or keep it in voxel space and step there.
            
            # Step in voxel space
            current_pos_vox += best_dir * (step_size / 1.0) # Assumes 1mm isotropic or handles via affine
            
            # Convert new voxel pos to world pos for the streamline list
            new_pos_world = (np.append(current_pos_vox, 1) @ affine.T)[:3]
            streamline.append(new_pos_world)
            
            prev_dir = best_dir

        if len(streamline) > 1:
            all_streamlines.append(np.array(streamline))
            
    return all_streamlines



In [29]:
seed_mask=(labels==2)
seeds= utils.seeds_from_mask(seed_mask, affine, density=vox_size)
csa_peaks=env.peaks


In [30]:
# Usage:
my_streamlines = custom_tracker(seeds, csa_peaks, affine)

In [31]:
len(my_streamlines)

3324

In [32]:
my_streamlines_actor= actor.line(
    my_streamlines, colors=colormap.line_colors(my_streamlines)
)

In [34]:
scene.add(my_streamlines_actor)
window.show(scene=scene,size=(800,800))

In [33]:
scene.clear()